# Fine-Tuning a Small Language Model for AI Poem Generation

This notebook trains the public `distilbert/distilgpt2` checkpoint on the small, original poem collection in `data/poems.csv`. It measures the untouched baseline and tuned model on the same held-out poems, compares generations for identical prompts, and saves the actual outputs. Run all cells in order. A Colab GPU is recommended; small-corpus results are illustrative, not a quality guarantee.

## 1. Choose a runtime and install the libraries
In Colab, choose **Runtime → Change runtime type → T4 GPU** if one is available. CPU also works but will be slower.

In [ ]:
!pip -q install transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 safetensors

## 2. Load the dataset
Upload `data/poems.csv` from the downloaded project folder when prompted. The poem column contains the text used for training.

In [ ]:
from google.colab import files
from pathlib import Path
import pandas as pd

uploaded = files.upload()  # Select data/poems.csv from the project folder
csv_name = next(name for name in uploaded if name.lower().endswith('.csv'))
df = pd.read_csv(csv_name)
df = df.dropna(subset=['poem']).copy()
print(f'Loaded {len(df)} poems')
display(df[['title', 'poem']].head(3))

## 3. Split the data and tokenize it
We keep 20% of poems aside for validation, using a fixed random seed so the split is repeatable. Causal language modeling learns to predict the next token in each poem.

In [ ]:
import random, numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling, set_seed)

SEED = 42
set_seed(SEED)
MODEL_NAME = 'distilbert/distilgpt2'
poems = df['poem'].astype(str).str.strip().tolist()
indices = list(range(len(poems)))
random.Random(SEED).shuffle(indices)
n_valid = max(1, round(len(indices) * 0.2))
valid_ids, train_ids = indices[:n_valid], indices[n_valid:]
train_texts = [poems[i] for i in train_ids]
valid_texts = [poems[i] for i in valid_ids]
print(f'Train poems: {len(train_texts)} | Validation poems: {len(valid_texts)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

train_ds = Dataset.from_dict({'text': train_texts}).map(tokenize_batch, batched=True, remove_columns=['text'])
valid_ds = Dataset.from_dict({'text': valid_texts}).map(tokenize_batch, batched=True, remove_columns=['text'])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 4. Measure the pretrained baseline
This is an evaluation only; no training has happened yet. The baseline perplexity is calculated as `exp(eval_loss)`.

In [ ]:
from math import exp

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id
eval_args = TrainingArguments(
    output_dir='baseline-eval', per_device_eval_batch_size=2,
    report_to='none', logging_strategy='no', save_strategy='no',
    evaluation_strategy='no'
)
baseline_trainer = Trainer(model=model, args=eval_args, eval_dataset=valid_ds,
                          data_collator=collator, tokenizer=tokenizer)
baseline_raw = baseline_trainer.evaluate()
baseline_loss = float(baseline_raw['eval_loss'])
baseline_ppl = exp(baseline_loss)
print({'baseline_eval_loss': baseline_loss, 'baseline_perplexity': baseline_ppl})

## 5. Generate baseline examples
We save outputs for fixed prompts, then run the same prompts with the fine-tuned model in Step 7. Identical decoding settings make the comparison easier to inspect.

In [ ]:
PROMPTS = ['Under the old tree,', 'At the edge of morning,', 'A small light in the window']
def generate_text(model, prompt, seed=123):
    set_seed(seed)
    encoded = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **encoded, max_new_tokens=70, do_sample=True, temperature=0.85,
            top_k=50, top_p=0.92, repetition_penalty=1.15,
            no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

baseline_generations = {prompt: generate_text(model, prompt) for prompt in PROMPTS}
for prompt, text in baseline_generations.items():
    print(f'\nPROMPT: {prompt}\n{text}\n')

## 6. Fine-tune the model
This calls `Trainer.train()` on the training poems. Five epochs are a short classroom starting point; this tiny dataset can overfit. Adjust epochs only if you have time to compare results.

In [ ]:
OUTPUT_DIR = 'poetry-distilgpt2-finetuned'
train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    logging_steps=5,
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=SEED
)
trainer = Trainer(model=model, args=train_args, train_dataset=train_ds,
                  eval_dataset=valid_ds, data_collator=collator, tokenizer=tokenizer)
train_result = trainer.train()  # This is the actual fine-tuning step
print(train_result.metrics)

## 7. Evaluate the tuned model and compare generations
Validation perplexity is useful for next-token prediction, but it is not a literary-quality score. Read the paired samples too.

In [ ]:
tuned_raw = trainer.evaluate()
tuned_loss = float(tuned_raw['eval_loss'])
tuned_ppl = exp(tuned_loss)
tuned_generations = {prompt: generate_text(trainer.model, prompt) for prompt in PROMPTS}

comparison = {prompt: {'pretrained': baseline_generations[prompt],
                        'fine_tuned': tuned_generations[prompt]} for prompt in PROMPTS}
for prompt, pair in comparison.items():
    print(f'\nPROMPT: {prompt}\n--- PRETRAINED ---\n{pair["pretrained"]}'
          f'\n--- FINE-TUNED ---\n{pair["fine_tuned"]}\n')

print(f'Baseline loss/perplexity: {baseline_loss:.4f} / {baseline_ppl:.4f}')
print(f'Fine-tuned loss/perplexity: {tuned_loss:.4f} / {tuned_ppl:.4f}')

## 8. Save the trained model and measured results
The model folder contains the tokenizer and fine-tuned weights. Download it or publish it to a model repository so the Streamlit app can load it.

In [ ]:
import json, shutil
from datetime import datetime, timezone

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
Path('results').mkdir(exist_ok=True)
metrics = {
    'run_time_utc': datetime.now(timezone.utc).isoformat(),
    'base_model': MODEL_NAME,
    'dataset_poems': len(poems),
    'train_poems': len(train_texts),
    'validation_poems': len(valid_texts),
    'epochs_requested': 5,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'baseline_eval_loss': baseline_loss,
    'baseline_perplexity': baseline_ppl,
    'finetuned_eval_loss': tuned_loss,
    'finetuned_perplexity': tuned_ppl,
    'train_metrics': {k: float(v) if isinstance(v, (int, float, np.number)) else str(v) for k, v in train_result.metrics.items()}
}
with open('results/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
with open('results/generation_comparison.json', 'w', encoding='utf-8') as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)
shutil.make_archive('poetry-distilgpt2-finetuned', 'zip', OUTPUT_DIR)
print('Saved model folder:', OUTPUT_DIR)
print('Saved metrics: results/metrics.json')
print('Saved paired examples: results/generation_comparison.json')
print('Saved downloadable model archive: poetry-distilgpt2-finetuned.zip')

## 9. Optional: upload the trained model to Hugging Face Hub
Only run this after creating a model repository and authenticating. Keep access tokens out of public notebooks and GitHub. For a public repo, the Streamlit app can download the model without a token.

In [ ]:
# Optional. Uncomment and set your own repository ID before running.
# from huggingface_hub import notebook_login
# notebook_login()
# MODEL_REPO_ID = 'YOUR_USERNAME/poetry-distilgpt2'
# trainer.model.push_to_hub(MODEL_REPO_ID)
# tokenizer.push_to_hub(MODEL_REPO_ID)

## 10. Use your measured results in the report
Download `results/metrics.json` and `results/generation_comparison.json`. Fill in `REPORT.md` using those exact values and examples. If evaluation fails, fix the run and do not substitute guessed numbers.